# Group B Dry Run
This notebook runs the small-sample Group B ASR smoke test for HW13. It is intended for VS Code + Colab extension debugging on an A100 before the full notebook is executed in the Colab Web UI.

Expected outputs:
- 1 GA19A dataset inspection printout
- 1 GA19B baseline transcription
- 4 Experiment 5A transcriptions (1 sample for each ASR model)
- 1 Experiment 5B sampling-rate-mismatch transcription
- Dry-run CSVs and log output

In [1]:
RUNNING_IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    RUNNING_IN_COLAB = True
    print('Running in Colab mode.')
except ImportError:
    print('google.colab not available; using local workspace mode.')

google.colab not available; using local workspace mode.


In [2]:
%pip install -q datasets transformers jiwer scipy soundfile librosa sentencepiece accelerate


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path
import os
import sys

DRIVE_ROOT = Path('/content/drive/MyDrive')
LOCAL_ROOT = Path('/Users/noir/visual_studio/Visual_Studio__UC_Spring_26/GEN_AI/HW1/kamp_hw13')
if RUNNING_IN_COLAB:
    CANDIDATE_ROOTS = [
        DRIVE_ROOT / 'kamp_hw13',
        DRIVE_ROOT / 'Visual_Studio__UC_Spring_26' / 'GEN_AI' / 'HW1' / 'kamp_hw13',
    ]
else:
    CANDIDATE_ROOTS = [
        LOCAL_ROOT,
        Path.cwd(),
        Path.cwd().parent,
    ]

def find_project_root():
    checked_paths = []
    for candidate in CANDIDATE_ROOTS:
        checked_paths.append(candidate)
        if (candidate / 'hw13_scripts' / 'kamp_hw13.py').exists():
            return candidate

    checked_display = '\n'.join(f' - {path}' for path in checked_paths)
    raise FileNotFoundError(
        'Could not locate kamp_hw13.\n'
        'In Colab, upload the project to /content/drive/MyDrive/kamp_hw13\n'
        'or mirror the local nesting under /content/drive/MyDrive/Visual_Studio__UC_Spring_26/GEN_AI/HW1/kamp_hw13.\n'
        'In local mode, open the notebook from the kamp_hw13 workspace.\n'
        f'Checked:\n{checked_display}'
    )

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'hw13_scripts'))

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'Module exists: {(PROJECT_ROOT / "hw13_scripts" / "kamp_hw13.py").exists()}')

PROJECT_ROOT = /Users/noir/visual_studio/Visual_Studio__UC_Spring_26/GEN_AI/HW1/kamp_hw13
Module exists: True


In [4]:
import importlib

import hw13_data_utils
import hw13_experiment_runner
import kamp_hw13

importlib.reload(hw13_data_utils)
importlib.reload(hw13_experiment_runner)
importlib.reload(kamp_hw13)

from hw13_data_utils import TeeLogger
from kamp_hw13 import run_group_b

DRY_RUN_ROOT = PROJECT_ROOT / 'hw13_experiments' / 'group_b_dry_run'
DRY_RUN_LOG = PROJECT_ROOT / 'hw13_printouts' / 'group_b_dry_run_log.txt'

print(f'Using shared run_group_b from {kamp_hw13.__file__}')
print(f'DRY_RUN_ROOT = {DRY_RUN_ROOT}')
print(f'DRY_RUN_LOG = {DRY_RUN_LOG}')

DRY_RUN_ROOT = /Users/noir/visual_studio/Visual_Studio__UC_Spring_26/GEN_AI/HW1/kamp_hw13/hw13_experiments/group_b_dry_run
DRY_RUN_LOG = /Users/noir/visual_studio/Visual_Studio__UC_Spring_26/GEN_AI/HW1/kamp_hw13/hw13_printouts/group_b_dry_run_log.txt


In [5]:
tee = TeeLogger(DRY_RUN_LOG)
tee_started = False

try:
    tee.start()
    tee_started = True
    print('[NOTEBOOK] Starting Group B dry run')
    summary = run_group_b(
        dry_run=True,
        experiments_dir=DRY_RUN_ROOT,
        checkpoint_enabled=False,
        resume=False,
    )
    print(summary)
finally:
    if tee_started:
        tee.stop()

[NOTEBOOK] Starting Group B dry run

[GROUP B] Starting dry-run ASR workflow
[GROUP B] Output root: /Users/noir/visual_studio/Visual_Studio__UC_Spring_26/GEN_AI/HW1/kamp_hw13/hw13_experiments/group_b_dry_run
[CSV] Initialized: /Users/noir/visual_studio/Visual_Studio__UC_Spring_26/GEN_AI/HW1/kamp_hw13/hw13_experiments/group_b_dry_run/exp1_baselines/exp1_baselines.csv
[CSV] Initialized: /Users/noir/visual_studio/Visual_Studio__UC_Spring_26/GEN_AI/HW1/kamp_hw13/hw13_experiments/group_b_dry_run/exp5_ga19b_asr/exp5_ga19b_asr.csv
/Users/noir/visual_studio/Visual_Studio__UC_Spring_26/GEN_AI/HW1/kamp_hw13/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
README.md: 23.5kB [00:00, 13.1MB/s]
en-AU/train-00000-of-00001.parquet: 100%|##########| 37.3M/37.3M [00:01<00:00, 23.2MB/s]
Generating train split: 100%|#

In [6]:
import csv

baseline_csv = DRY_RUN_ROOT / 'exp1_baselines' / 'exp1_baselines.csv'
exp5_csv = DRY_RUN_ROOT / 'exp5_ga19b_asr' / 'exp5_ga19b_asr.csv'

assert baseline_csv.exists(), f'Missing dry-run baseline CSV: {baseline_csv}'
assert exp5_csv.exists(), f'Missing dry-run Exp 5 CSV: {exp5_csv}'
assert DRY_RUN_LOG.exists(), f'Missing dry-run log: {DRY_RUN_LOG}'

with exp5_csv.open() as handle:
    rows = list(csv.DictReader(handle))

models = sorted({row['model_name'] for row in rows if row.get('model_name')})
print(f'Exp 5 dry-run rows: {len(rows)}')
print(f'Exp 5 models: {models}')

assert len(rows) >= 5, f'Expected at least 5 Exp 5 dry-run rows, found {len(rows)}'
assert len(models) >= 4, f'Expected at least 4 ASR models in dry run, found {models}'
print('Dry-run smoke test completed successfully.')

Exp 5 dry-run rows: 5
Exp 5 models: ['facebook/wav2vec2-base-960h', 'openai/whisper-medium', 'openai/whisper-small', 'openai/whisper-tiny']
Dry-run smoke test completed successfully.
